> 在线讲义：[第 24 章 · 收尾：数参数、warmup 学习率、对照原文通读 §3](https://wikieden.github.io/transformer-visual/lessons/24-wrap-up.html) · 本书《从零读懂 Transformer》

# 第 24 章 · 收尾：数参数、warmup 学习率、对照原文通读 §3

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/24-wrap-up.html) · 仓库里 `transformer-deep-dive/notes/lessons/24-wrap-up.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 一个 attention 子层 | $4d_{model}^2$ | `n_params(nn.MultiheadAttention(4, 2, bias=False))` | 64 |
| 一个 FFN 子层 | $2 d_{model} d_{ff}$ | 两个 `nn.Linear(bias=False)` | 128 |
| encoder 层 / decoder 层 | 1 个 / 2 个 attention + FFN | `n_params(...)` | 192 / 256 |
| embedding 表 | $V \cdot d_{model}$，三处共用 | `lm_head.weight = emb.weight` | 32，只算一次 |
| 整个玩具模型 | 2 × 192 + 2 × 256 + 32 | `n_params(toy)` | 928 |
| 位置编码没有参数 | 固定的 sin、cos | `register_buffer` | 在 buffers 里，不算参数 |
| warmup 学习率 | Eq. 3 | `lrate(step)` | 1000、4000、100000 步：0.000175、0.000699、0.000140 |
| PyTorch 调度器 | 同一条曲线 | `LambdaLR` + `Adam(betas=(0.9, 0.98), eps=1e-9)` | 最高点在第 4000 步 |

只算讲义里的玩具模型（d_model = 4、h = 2、d_ff = 16、V = 8、N = 2）和 step = 1000、4000、100000。
题 1、题 2 做完以后，可以调用 count_params(...) 和 lrate(...) 自己验证答案。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
from torch import nn

**类 `Toy`**：只放"要学的矩阵"：attention 的 W^Q W^K W^V W^O、FFN 的 W1 W2、一张共享 embedding 表。不要 bias 和 LayerNorm。

In [ ]:
class Toy(nn.Module):
    """只放"要学的矩阵"：attention 的 W^Q W^K W^V W^O、FFN 的 W1 W2、一张共享 embedding 表。不要 bias 和 LayerNorm。"""

    def __init__(self, d: int, h: int, d_ff: int, vocab: int, n_layers: int):
        super().__init__()

        def attn():
            return nn.MultiheadAttention(d, h, bias=False)

        def ffn():
            return nn.Sequential(nn.Linear(d, d_ff, bias=False), nn.Linear(d_ff, d, bias=False))

        self.enc = nn.ModuleList(nn.ModuleDict({"attn": attn(), "ffn": ffn()}) for _ in range(n_layers))
        self.dec = nn.ModuleList(
            nn.ModuleDict({"self_attn": attn(), "cross_attn": attn(), "ffn": ffn()}) for _ in range(n_layers)
        )
        self.emb = nn.Embedding(vocab, d)
        self.lm_head = nn.Linear(d, vocab, bias=False)
        self.lm_head.weight = self.emb.weight  # 三处共用一张表（第 22 章）
        self.register_buffer("pe", torch.zeros(64, d))  # 位置编码是固定的 sin / cos：buffer，不是参数

**函数 `n_params`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def n_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())  # 共用的参数只会被列出一次

**函数 `count_params`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def count_params(d: int, h: int, d_ff: int, vocab: int, n_layers: int) -> dict[str, int]:
    toy = Toy(d, h, d_ff, vocab, n_layers)
    return {
        "attention 子层": n_params(toy.enc[0]["attn"]),
        "FFN 子层": n_params(toy.enc[0]["ffn"]),
        "encoder 层": n_params(toy.enc[0]),
        "decoder 层": n_params(toy.dec[0]),
        "embedding 表": n_params(toy.emb),
        "整个模型": n_params(toy),
    }

**函数 `lrate`**：Eq. 3：d^-0.5 · min(step^-0.5, step · warmup^-1.5)。step 从 1 开始（0 的 −0.5 次方是无穷大）。

In [ ]:
def lrate(step: int, d_model: int = 512, warmup: int = 4000) -> float:
    """Eq. 3：d^-0.5 · min(step^-0.5, step · warmup^-1.5)。step 从 1 开始（0 的 −0.5 次方是无穷大）。"""
    step = max(step, 1)
    return d_model**-0.5 * min(step**-0.5, step * warmup**-1.5)

**函数 `scheduler_demo`**：用 PyTorch 的 LambdaLR 实现同一条曲线：基础 lr 设 1，每步乘 lrate(step)。

In [ ]:
def scheduler_demo(d_model: int = 512, warmup: int = 4000, steps: int = 6000) -> list[float]:
    """用 PyTorch 的 LambdaLR 实现同一条曲线：基础 lr 设 1，每步乘 lrate(step)。"""
    w = nn.Parameter(torch.zeros(1))
    opt = torch.optim.Adam([w], lr=1.0, betas=(0.9, 0.98), eps=1e-9)  # 论文 §5.3 L365
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: lrate(s + 1, d_model, warmup))
    lrs = []
    for _ in range(steps):
        lrs.append(opt.param_groups[0]["lr"])
        opt.step()
        sched.step()
    return lrs

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("玩具模型（d_model = 4、h = 2、d_ff = 16、V = 8、N = 2），PyTorch 自己数:")
for name, k in count_params(4, 2, 16, 8, 2).items():
    print(f"  {name:<12} {k}")
toy = Toy(4, 2, 16, 8, 2)
print("embedding 表和输出层是同一个参数:", toy.lm_head.weight is toy.emb.weight, "→ 只算一次")
print("位置编码在 buffers 里、不在 parameters 里:", "pe" in dict(toy.named_buffers()))

In [ ]:
print("\nwarmup 学习率（d_model = 512，warmup_steps = 4000）")
for s in [1000, 4000, 100000]:
    print(f"  step {s:>6}: {lrate(s):.6f}")
lrs = scheduler_demo()
peak = max(range(len(lrs)), key=lrs.__getitem__) + 1
print(f"LambdaLR 跑 6000 步：最高点在第 {peak} 步，学习率 {max(lrs):.6f}")

## 3. 代码里才看得到的

1. **`model.parameters()` 自动去重**：输出层和 embedding 指向同一个参数，`parameters()` 只列一次，所以直接加起来就是"共享只算一次"的结果，不用自己减。
2. **buffer（缓存）不是参数**：位置编码用 `register_buffer` 存，会跟着模型保存、搬到 GPU，但不参与训练，也不算进参数量。
3. **`LambdaLR`**：优化器的基础学习率设成 1，每一步乘上 `lrate(step)`，就得到论文那条先升后降的曲线。step 要从 1 开始：0 的 −0.5 次方是无穷大。
4. **论文的 Adam 设置**（§5.3 L365）：β₁ = 0.9、β₂ = 0.98、ε = $10^{-9}$，代码里照抄。

## 4. 自己改一改

- 题 1 做完以后验证：`uv run python -c "from ch24_wrap_up import count_params; print(count_params(512, 8, 2048, 37000, 6))"`，和你算的、和 Table 3 的 65M 对一对。
- 题 2 做完以后验证：`uv run python -c "from ch24_wrap_up import lrate; print(lrate(2000) / lrate(4000))"`，16000 步同理。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 数参数，分零件：attention 四 d 方，FFN 二 d d_ff；
> 编码一个 attention，解码两个，乘层数再相加；embedding 一张表，位置编码不花钱。
> 学习率是步子：先直线升四千步，再按根号慢慢降；开头热身别踩崩。
> 通读 §3 按地图：一段原文一段复述，对上章节再往下。